# 資料前處理 Data Preprocessing

In [122]:
# 確認 Notebook 使用的是專案環境，並載入本課需要的套件。
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib
from pathlib import Path

pd.set_option("display.max_columns", 20)
print("Python:", sys.version.split()[0])
print("執行環境:", sys.executable)
print("pandas:", pd.__version__, "sklearn:", sklearn.__version__)


Python: 3.12.13
執行環境: d:\github\python-machine-learning\.venv\Scripts\python.exe
pandas: 2.3.3 sklearn: 1.9.1


## 查看資料集

情境：行銷部想在聯絡客戶前，預測客戶是否會購買。

`purchased` 是結果；其他資料是聯絡前已知的資訊。合成資料的分數只能展示流程，不能當作真實商業成效。

| 欄位 | 意義 | 用途 |
|---|---|---|
| customer_id | 客戶識別碼 | 對應資料，不作為模型特徵 |
| age | 年齡（歲） | 數值特徵 |
| monthly_income | 月收入（元） | 數值特徵 |
| website_visits | 最近一個月網站造訪次數 | 數值特徵 |
| city | 居住城市 | 無順序類別 |
| channel | 接觸管道 | 無順序類別 |
| membership | Basic / Silver / Gold 會員等級 | 有順序類別；後面示範兩種編碼選擇 |
| purchased | Yes / No，後續是否購買 | Target |


In [123]:
from pathlib import Path

### 設定專案路徑
project_root = Path("../data_preprocessing")
df = pd.read_csv(project_root / "data" / "customers.csv")

print("資料形狀（列數、欄數）:", df.shape)
display(df.info())
display(df.head(8))



資料形狀（列數、欄數）: (240, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   customer_id     240 non-null    object 
 1   age             219 non-null    float64
 2   monthly_income  216 non-null    float64
 3   website_visits  234 non-null    float64
 4   city            233 non-null    object 
 5   channel         222 non-null    object 
 6   membership      230 non-null    object 
 7   purchased       240 non-null    object 
dtypes: float64(3), object(5)
memory usage: 15.1+ KB


None

,customer_id,age,monthly_income,website_visits,city,channel,membership,purchased
0,C0001,60.0,38000.0,0.0,Tainan,NaN,Basic,No
1,C0002,52.0,101000.0,0.0,Tainan,Ads,NaN,Yes
2,C0003,30.0,NaN,5.0,Taichung,Ads,Basic,Yes
3,C0004,54.0,39000.0,6.0,Taipei,Referral,Silver,No
4,C0005,34.0,61000.0,1.0,Taipei,Ads,Silver,No
5,C0006,58.0,105000.0,2.0,Tainan,Referral,Basic,Yes
6,C0007,23.0,53000.0,0.0,Taichung,Organic,Silver,No
7,C0008,29.0,57000.0,2.0,Taipei,Referral,Gold,No


## 資料清洗：查看缺失值

空值數量、比例、數值摘要與目標分布，是接手資料時的基本檢查

In [124]:
### 檢查缺失值
display(pd.DataFrame({
    "缺失筆數": df.isna().sum(), # 計算每個欄位有幾筆缺失值
    "缺失比例": df.isna().mean() # 計算每個欄位缺失值所占的比例
}).round(3)) # 將結果四捨五入到小數第 3 位

,缺失筆數,缺失比例
customer_id,0,0.000
age,21,0.088
monthly_income,24,0.100
website_visits,6,0.025
city,7,0.029
channel,18,0.075
membership,10,0.042
purchased,0,0.000


In [125]:
### 查看目標欄位的資料分布
display(df["purchased"].value_counts(dropna=False)) # 統計 purchased 各類別出現次數，包含缺失值

purchased
Yes    124
No     116
Name: count, dtype: int64

In [126]:
### 檢查客戶 ID 是否重複
print("重複客戶 ID:",df["customer_id"].duplicated().sum()) # 檢查 customer_id 是否重複，避免同一位客戶出現多次

重複客戶 ID: 0


In [127]:
### 檢查完全重複的資料
print("完全重複列:",df.duplicated().sum()) # 計算整列內容完全相同的資料筆數

完全重複列: 0


### 常見髒資料：空白、文字數字、負值

> 以下使用獨立小表，不會改動主資料。

正式資料常用空字串、`?`、`N/A` 表示缺失。先約定什麼代表空值，再清理。

清除格式與依業務規則檢查負值，不需要學習全體統計量。


In [128]:
### 做一個贓資料
dirty = pd.DataFrame({"age": [" 25 ", "?", "-3"], "city": [" Taipei ", "", "Taichung"]})
display(dirty)

,age,city
0,25,Taipei
1,?,
2,-3,Taichung


In [129]:
clean = dirty.copy() # 複製原始資料，避免直接修改 dirty
clean["city"] = clean["city"].str.strip().replace("", np.nan) # 移除城市名稱前後空白，並將空字串轉成缺失值 NaN
clean["age"] = pd.to_numeric(clean["age"], errors="coerce") # 將年齡轉成數值，無法轉換的內容自動變成 NaN
clean.loc[clean["age"] < 0, "age"] = np.nan # 年齡不可能為負數，將不合理的負值改成 NaN，留待後續處理
display(clean)

,age,city
0,25.0,Taipei
1,NaN,NaN
2,NaN,Taichung


## Features 與 Target：先問「要預測什麼」

- **Features（X）** 是拿來預測的輸入欄位；
- **Target（y）** 是希望模型學會預測的結果。
- X 通常是二維表格，y 是一維序列。
- 每列 X 必須對應同一位客戶的 y。

不要放入 `purchased` 本身，也不要放入購買後才知道的付款金額、出貨狀態等欄位。
這些會讓練習分數漂亮，卻無法在真正預測時使用，稱為**資料洩漏**。
ID 在本情境只用來識別客戶，因此排除。

![](./images/Features與Target.png)

In [130]:
### 選擇 Features (輸入特徵)
X = df[[
    "age",
    "monthly_income",
    "website_visits",
    "city",
    "channel",
    "membership"
]]

### 選擇 Target (預測目標)
y = df["purchased"]

### 查看資料
print("X:", X.shape)
print("y:", y.shape)

display(X.head())
display(y.head())

X: (240, 6)
y: (240,)


,age,monthly_income,website_visits,city,channel,membership
0,60.0,38000.0,0.0,Tainan,NaN,Basic
1,52.0,101000.0,0.0,Tainan,Ads,NaN
2,30.0,NaN,5.0,Taichung,Ads,Basic
3,54.0,39000.0,6.0,Taipei,Referral,Silver
4,34.0,61000.0,1.0,Taipei,Ads,Silver


0     No
1    Yes
2    Yes
3     No
4     No
Name: purchased, dtype: object

## Train / Test Split：留一些沒看過的題目

訓練集用來學習，測試集用來檢查對未見資料的表現。

先切分，再補值、編碼、縮放。否則處理規則可能偷偷包含測試資料的資訊。

| 參數 | 本課設定 | 意義 |
|---|---|---|
| test_size | 0.2 | 保留約 20% 作為測試集 |
| random_state | 42 | 固定切分結果，方便重現 |
| stratify | y | 分類題盡量維持各類答案比例 |


In [131]:
# 同時切 X 與 y，讓列的對應關係保持一致。
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y # 見下面補充
)

print("訓練集:", X_train.shape, "測試集:", X_test.shape)

display(pd.DataFrame({
    "全部": y.value_counts(normalize=True), # 計算完整資料中各類別所占的比例
    "訓練": y_train.value_counts(normalize=True), # 計算訓練集中各類別所占的比例
    "測試": y_test.value_counts(normalize=True), # 計算測試集中各類別所占的比例
}).round(3)) # 將結果整理成 DataFrame，並四捨五入至小數點後 3 位


訓練集: (192, 6) 測試集: (48, 6)


,全部,訓練,測試
purchased,,,
Yes,0.517,0.516,0.521
No,0.483,0.484,0.479


### 補充：`stratify=y` 

stratify=y 是在切訓練集、測試集時，要求它們盡量維持原本 y 的類別比例。

例如原始資料的 y 有 100 人：(有購賣80人，沒購買20人)

加上stratify=y，它會盡量切成
```text
              有購買    沒購買
全部資料        80%      20%
訓練資料        80%      20%
測試資料        80%      20%
```

如果沒有 stratify=y，train_test_split() 預設是隨機切，所以可能變成：

```text
              有購買    沒購買
全部資料        80%      20%
訓練資料        84%      16%
測試資料        64%      36%
```

資料量很大時差異可能不明顯，但資料量小或類別不平衡時特別容易出問題。

## 缺失值處理：SimpleImputer

先知道缺失原因，再選策略。

- 數值類別以**中位數**補
- 文字類別以**最常見類別(眾數)**補

中位數比較不容易被極端值拉動，但補值不會恢復真實答案。

| 策略 | 常用情境 | 注意 |
|---|---|---|
| mean | 數值分布較平穩 | 容易受極端值影響 |
| median | 收入等可能有極端值 | 本課數值預設 |
| most_frequent | 類別欄位 | 可能強化多數類別 |
| constant | 用 Unknown 或固定值 | 數值補 0 必須有業務意義 |
| 刪除列 | 少量且可合理捨棄 | 大量刪除會損失資料、改變分布 |


### 處理數值空值

In [132]:
from sklearn.impute import SimpleImputer
import pandas as pd


### 建立中位數補值器
num_imputer = SimpleImputer(
    strategy="median" # 使用中位數填補缺失值
)


### 處理訓練集
X_train[["age", "monthly_income", "website_visits"]] = num_imputer.fit_transform(
    X_train[["age", "monthly_income", "website_visits"]]
)


### 處理測試集
X_test[["age", "monthly_income", "website_visits"]] = num_imputer.transform(
    X_test[["age", "monthly_income", "website_visits"]]
)

### 處理類別空值

In [133]:
from sklearn.impute import SimpleImputer


### 建立類別資料補值器
cat_imputer = SimpleImputer(
    strategy="most_frequent", # 使用眾數填補缺失值
    keep_empty_features=True # 即使整個欄位都是空值，也保留該欄位
)


### 處理訓練集
X_train[["city", "channel", "membership"]] = cat_imputer.fit_transform(
    X_train[["city", "channel", "membership"]]
)


### 處理測試集
X_test[["city", "channel", "membership"]] = cat_imputer.transform(
    X_test[["city", "channel", "membership"]]
)

## 類別資料編碼：先分清楚欄位用途

模型並不能處理文字類別，所以我們要將文字類別換成數字代碼，讓模型能處理。

![](./images/三種encoder比較.png)

### Label Encoding：把答案名稱換成代碼

`LabelEncoder` 適合 **Target（y）**。例如 No、Yes 對應 0、1。

代碼只代表名稱，不表示好壞或數量。不要將城市用 LabelEncoder 編成 0、1、2 再當連續數值使用，模型可能誤以為城市有大小距離。


In [134]:
### 建立 Label Encoder
label_encoder = LabelEncoder() # 建立標籤編碼器，將文字類別轉換成數字代碼

### 編碼
y_train_encoded = label_encoder.fit_transform(y_train) # 編碼訓練集答案
y_test_encoded = label_encoder.transform(y_test) # 編碼測試集答案

In [135]:
### 查看 Label Encoder 建立的編碼規則
display(
    pd.DataFrame({
        "原始標籤": label_encoder.classes_, # Label Encoder 從訓練集找到的所有類別
        "代碼": range(len(label_encoder.classes_)) # 每個類別所對應的數字代碼
    })
)

,原始標籤,代碼
0,No,0
1,Yes,1


In [136]:
### 比較編碼前與編碼後
print("編碼前:", y_train.iloc[:8].tolist()) # 查看前 8 筆原始文字標籤
print("編碼後:", y_train_encoded[:8]) # 查看前 8 筆轉換後的數字代碼

編碼前: ['No', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes']
編碼後: [0 1 1 0 1 0 1 1]


In [137]:
### 將數字代碼還原成原始標籤
print("還原:", label_encoder.inverse_transform(y_train_encoded[:8])) # 將數字重新轉回原本的文字類別

還原: ['No' 'Yes' 'Yes' 'No' 'Yes' 'No' 'Yes' 'Yes']


### One-Hot Encoding(獨熱編碼)


假設原始資料有一欄：

```text
city
------
台北
高雄
台南
台北
```

我們不能很隨便地寫成：

```text
台北 → 0
高雄 → 1
台南 → 2
```

因為某些模型看到 `0、1、2`，可能會把它當成具有數值關係：台南(2) > 高雄(1) > 台北(0)

但「城市」根本沒有這種大小與距離關係。所以 One-Hot Encoding 的做法是：**一個類別拆成一個欄位。**

```text
原本：

city
------
台北
高雄
台南
台北


One-Hot Encoding 後：

        city_台北   city_台南   city_高雄
台北        1          0          0
高雄        0          0          1
台南        0          1          0
台北        1          0          0
```

![](./images/獨熱編碼.png)

In [138]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder


### 指定需要 One-Hot Encoding 的欄位
columns = ["city", "channel"]

### 建立 One-Hot Encoder
onehot = OneHotEncoder(
    handle_unknown="ignore", # 遇到沒看過的新類別時不報錯
    sparse_output=False # 輸出一般陣列
)


### 編碼訓練集
train_encoded = pd.DataFrame(
    onehot.fit_transform(X_train[columns]),
    columns=onehot.get_feature_names_out(columns),
    index=X_train.index
)


### 編碼測試集
test_encoded = pd.DataFrame(
    onehot.transform(X_test[columns]),
    columns=onehot.get_feature_names_out(columns),
    index=X_test.index
)

In [139]:
### 查看編碼前後的差異
print("編碼前：")
display(X_train[columns].head(3))

print("編碼後：")
display(train_encoded.head(3))

編碼前：


,city,channel
44,Taipei,Organic
187,Tainan,Referral
130,Taichung,Organic


編碼後：


,city_Taichung,city_Tainan,city_Taipei,channel_Ads,channel_Organic,channel_Referral
44,0.0,0.0,1.0,0.0,1.0,0.0
187,0.0,1.0,0.0,0.0,0.0,1.0
130,1.0,0.0,0.0,0.0,1.0,0.0


In [140]:
### 刪除原本的類別欄位，合併編碼後的欄位
X_train = pd.concat(
    [X_train.drop(columns=columns), train_encoded],
    axis=1
)

X_test = pd.concat(
    [X_test.drop(columns=columns), test_encoded],
    axis=1
)


### 查看處理後的資料
display(X_train.head())
display(X_test.head())

,age,monthly_income,website_visits,membership,city_Taichung,city_Tainan,city_Taipei,channel_Ads,channel_Organic,channel_Referral
44,56.0,72000.0,7.0,Silver,0.0,0.0,1.0,0.0,1.0,0.0
187,38.0,72000.0,7.0,Silver,0.0,1.0,0.0,0.0,0.0,1.0
130,42.5,29000.0,10.0,Silver,1.0,0.0,0.0,0.0,1.0,0.0
32,42.5,66000.0,0.0,Basic,0.0,0.0,1.0,0.0,1.0,0.0
34,62.0,49000.0,5.0,Gold,1.0,0.0,0.0,1.0,0.0,0.0


,age,monthly_income,website_visits,membership,city_Taichung,city_Tainan,city_Taipei,channel_Ads,channel_Organic,channel_Referral
35,64.0,61000.0,8.0,Silver,0.0,0.0,1.0,1.0,0.0,0.0
90,42.5,77000.0,10.0,Gold,0.0,0.0,1.0,0.0,1.0,0.0
46,59.0,92000.0,0.0,Gold,1.0,0.0,0.0,0.0,0.0,1.0
150,44.0,102000.0,6.0,Silver,0.0,0.0,1.0,1.0,0.0,0.0
28,39.0,70000.0,12.0,Basic,0.0,0.0,1.0,0.0,1.0,0.0


### 有順序的 Features：OrdinalEncoder
會員等級確定有 Basic → Silver → Gold 的順序時，可以用 `OrdinalEncoder` 手動指定順序。

In [141]:
from sklearn.preprocessing import OrdinalEncoder

### 建立 Ordinal Encoder
ordinal = OrdinalEncoder(
    categories=[["Basic", "Silver", "Gold"]], # 指定會員等級順序
    handle_unknown="use_encoded_value", # 允許出現未知類別
    unknown_value=-1 # 未知類別編碼為 -1
)

### 查看編碼前的資料
print("編碼前：")
display(X_train[["membership"]].head())


### 編碼訓練集
X_train[["membership"]] = ordinal.fit_transform(
    X_train[["membership"]]
)


### 編碼測試集
X_test[["membership"]] = ordinal.transform(
    X_test[["membership"]]
)


### 查看編碼後的資料
print("編碼後：")
display(X_train[["membership"]].head())

編碼前：


,membership
44,Silver
187,Silver
130,Silver
32,Basic
34,Gold


編碼後：


,membership
44,1.0
187,1.0
130,1.0
32,0.0
34,2.0


## Feature Scaling(特徵縮放)：讓數值尺度容易一起使用

不同 Feature 的數值範圍可能差非常多。例如一份客戶資料中：

| Feature | 範例數值 |
|---|---:|
| 年齡 | 25、32、45 |
| 年收入 | 40000、60000、80000 |
| 每月造訪次數 | 2、5、8 |

雖然這三個欄位都屬於數值資料，但它們的尺度完全不同。

假設有兩位客戶：

```text
客戶 A：年收入 40,000、造訪次數 3
客戶 B：年收入 60,000、造訪次數 8

# 兩人的差距為
收入差距：20,000
造訪次數差距：5
```

如果演算法直接拿原始數值計算，20,000 遠大於 5，「收入」很容易在計算過程中佔據較大的影響。

但這不代表「收入一定比造訪次數重要 4,000 倍」。

Feature Scaling就是把不同數值 Feature 轉換到比較容易一起計算的尺度，同時保留資料原本的相對關係。

![](./images/特徵縮放.png)

![](./images/比較Standardization、Normalization%20和%20MinMaxScaler.png)


### Standardization(標準化)
把每個數值欄位移到平均約 0、標準差約 1 的尺度。**不會把資料變成常態分布**，也不保證值在 0～1。

規則從訓練集學習，因此測試集的平均值不需要剛好是 0。


In [144]:
from sklearn.preprocessing import StandardScaler

### 選擇需要標準化的數值欄位
numeric_features = ["age", "monthly_income", "website_visits"]


### 取得訓練集與測試集的數值資料
train_num = X_train[numeric_features].copy()
test_num = X_test[numeric_features].copy()


### 建立 StandardScaler
standard_scaler = StandardScaler() # 將數值資料轉換成平均數約 0、標準差約 1


### 標準化訓練集
train_num_scaled = pd.DataFrame(
    standard_scaler.fit_transform(train_num), # 學習訓練集的平均數與標準差，並進行標準化
    columns=numeric_features, # 保留數值欄位名稱
    index=X_train.index # 保留訓練集原本的索引
)


### 標準化測試集
test_num_scaled = pd.DataFrame(
    standard_scaler.transform(test_num), # 使用訓練集學到的規則進行標準化
    columns=numeric_features, # 保留數值欄位名稱
    index=X_test.index # 保留測試集原本的索引
)


### 比較標準化前與標準化後
display(train_num.head())
display(train_num_scaled.head().round(3))

,age,monthly_income,website_visits
44,56.0,72000.0,7.0
187,38.0,72000.0,7.0
130,42.5,29000.0,10.0
32,42.5,66000.0,0.0
34,62.0,49000.0,5.0


,age,monthly_income,website_visits
44,1.067,0.129,0.328
187,-0.390,0.129,0.328
130,-0.026,-1.714,1.126
32,-0.026,-0.128,-1.533
34,1.553,-0.856,-0.204


### Normalization(正規化)：也稱 Min-Max Scaling

> 許多入門教材把縮放至 0～1 稱為 Normalization，本課使用 `MinMaxScaler` 示範這個意思。

訓練資料各欄最小值映到 0、最大值映到 1；新資料超出原範圍時，結果可以小於 0 或大於 1。

`clip=True` 可限制輸出範圍，但會讓超出範圍的程度無法保留，不能直接當成資料品質修復。


In [146]:
from sklearn.preprocessing import MinMaxScaler


### 選擇需要正規化的數值欄位
numeric_features = ["age", "monthly_income", "website_visits"]


### 取得訓練集與測試集的數值資料
train_num = X_train[numeric_features].copy()
test_num = X_test[numeric_features].copy()


### 建立 MinMaxScaler
minmax_scaler = MinMaxScaler() # 建立 Min-Max 縮放器，預設將數值縮放到 0～1


### 正規化訓練集
train_num_minmax = pd.DataFrame(
    minmax_scaler.fit_transform(train_num),
    columns=numeric_features, # 補回原本的數值欄位名稱
    index=X_train.index # 保留訓練集原本的索引
)


### 正規化測試集
test_num_minmax = pd.DataFrame(
    minmax_scaler.transform(test_num),
    columns=numeric_features, # 使用與訓練集相同的欄位名稱
    index=X_test.index # 保留測試集原本的索引
)


### 比較正規化前與正規化後
display(train_num.head())
display(train_num_minmax.head().round(3))

,age,monthly_income,website_visits
44,56.0,72000.0,7.0
187,38.0,72000.0,7.0
130,42.5,29000.0,10.0
32,42.5,66000.0,0.0
34,62.0,49000.0,5.0


,age,monthly_income,website_visits
44,0.800,0.558,0.583
187,0.400,0.558,0.583
130,0.500,0.058,0.833
32,0.500,0.488,0.000
34,0.933,0.291,0.417


## 建立 Logistic Regression 分類模型

![](./images/機器學習流程圖.png)


In [151]:
### 建立模型使用的 Features
X_train_ready = X_train.copy()
X_test_ready = X_test.copy()


### 使用正規化後的數值取代原始數值
X_train_ready[numeric_features] = train_num_minmax
X_test_ready[numeric_features] = test_num_minmax


### 查看處理完成的資料
display(X_train_ready.head())
display(X_test_ready.head())

,age,monthly_income,website_visits,membership,city_Taichung,city_Tainan,city_Taipei,channel_Ads,channel_Organic,channel_Referral
44,0.800000,0.558140,0.583333,1.0,0.0,0.0,1.0,0.0,1.0,0.0
187,0.400000,0.558140,0.583333,1.0,0.0,1.0,0.0,0.0,0.0,1.0
130,0.500000,0.058140,0.833333,1.0,1.0,0.0,0.0,0.0,1.0,0.0
32,0.500000,0.488372,0.000000,0.0,0.0,0.0,1.0,0.0,1.0,0.0
34,0.933333,0.290698,0.416667,2.0,1.0,0.0,0.0,1.0,0.0,0.0


,age,monthly_income,website_visits,membership,city_Taichung,city_Tainan,city_Taipei,channel_Ads,channel_Organic,channel_Referral
35,0.977778,0.430233,0.666667,1.0,0.0,0.0,1.0,1.0,0.0,0.0
90,0.500000,0.616279,0.833333,2.0,0.0,0.0,1.0,0.0,1.0,0.0
46,0.866667,0.790698,0.000000,2.0,1.0,0.0,0.0,0.0,0.0,1.0
150,0.533333,0.906977,0.500000,1.0,0.0,0.0,1.0,1.0,0.0,0.0
28,0.422222,0.534884,1.000000,0.0,0.0,0.0,1.0,0.0,1.0,0.0


In [152]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


### 建立 Logistic Regression 模型
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)


### 訓練模型
model.fit(X_train_ready, y_train_encoded)


### 使用測試集進行預測
y_pred = model.predict(X_test_ready)


### 評估模型準確率
print("Accuracy：", round(accuracy_score(y_test_encoded, y_pred), 3))


### 比較前 10 筆真實答案與預測結果
df_result = pd.DataFrame({
    "真實答案": y_test_encoded[:10],
    "預測結果": y_pred[:10]
})

display(df_result)

Accuracy： 0.792


,真實答案,預測結果
0,0,1
1,1,1
2,0,0
3,1,1
4,1,1
5,1,1
6,1,1
7,1,0
8,1,1
9,0,0


## 正式工作建議：ColumnTransformer + Pipeline

手動流程適合學習，但新資料也必須完整重做一樣的步驟，容易漏掉。
- **ColumnTransformer** 按欄位分工；
- **Pipeline** 把步驟依序包起來。

對完整 Pipeline 呼叫 `fit(X_train, y_train)`，就能依序學到補值、編碼、縮放與模型。

呼叫 `predict(X_test)` 時會自動沿用已學到的規則。

流程：
- 數值欄位：中位數補值 → StandardScaler。
- 類別欄位：眾數補值 → OneHotEncoder。
- 合併處理結果 → LogisticRegression。

這次直接用文字 y，讓整個預測流程直接回傳 Yes / No，避免交付時漏帶 LabelEncoder。


In [154]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


### 重新切分原始資料，避免使用前面已經編碼的 X_train
X_train_pipe, X_test_pipe, y_train_pipe, y_test_pipe = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


### 指定不同類型的 Features
numeric_features = ["age", "monthly_income", "website_visits"]
categorical_features = ["city", "channel"]
ordinal_features = ["membership"]


### 數值資料：缺失值填補 + 標準化
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


### 無順序類別：缺失值填補 + One-Hot Encoding
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


### 有順序類別：缺失值填補 + Ordinal Encoding
ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=[["Basic", "Silver", "Gold"]],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])


### 整合不同欄位的前處理流程
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
    ("ordinal", ordinal_pipeline, ordinal_features)
])


### 建立完整 Pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])


### 訓練模型
model.fit(X_train_pipe, y_train_pipe)


### 使用測試集進行預測
y_pred = model.predict(X_test_pipe)


### 評估模型準確率
print("Accuracy：", round(accuracy_score(y_test_pipe, y_pred), 3))


### 比較前 10 筆真實答案與預測結果
df_result = pd.DataFrame({
    "真實答案": y_test_pipe.iloc[:10].to_numpy(),
    "預測結果": y_pred[:10]
})

display(df_result)

Accuracy： 0.792


,真實答案,預測結果
0,No,Yes
1,Yes,Yes
2,No,No
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,No
8,Yes,Yes
9,No,No


### 評估模型

- Accuracy 是答對比例。若大多數客戶不買，全部猜 No 也可能有高分。
- precision 看預測某類時多可靠
- recall 看實際該類找回多少
- F1 綜合兩者




In [156]:
from sklearn.metrics import classification_report


### 評估分類模型
print(classification_report(
    y_test_pipe, # 測試集的真實答案
    y_pred, # 模型預測的答案
    digits=3, # 評估指標顯示至小數點後 3 位
    zero_division=0 # 避免模型沒有預測某類別時出現除以零的警告
))

              precision    recall  f1-score   support

          No      0.842     0.696     0.762        23
         Yes      0.759     0.880     0.815        25

    accuracy                          0.792        48
   macro avg      0.800     0.788     0.788        48
weighted avg      0.799     0.792     0.789        48



如何解讀：

1. Precision (精確率)：當模型預測某個類別時，有多少比例預測正確？
	- No = 84.2%：模型預測不購買的客戶中，84.2% 真的沒有購買。
	- Yes = 75.9%：模型預測會購買的客戶中，75.9% 真的有購買。
	- 這代表模型預測 No 時比較可靠。
2. Recall (召回率)：實際屬於某個類別的客戶，模型成功找出多少？
	- No = 69.6%：23 位實際不購買的客戶，模型成功找出 16 位。
	- Yes = 88.0%：25 位實際會購買的客戶，模型成功找出 22 位。
	- 這代表模型比較擅長找出會購買的客戶。
3. F1-score (F1 分數)：F1-score 綜合考量 Precision 與 Recall。
	- No = 76.2%
	- Yes = 81.5%
4. Support (樣本數)：Support 是測試集中各類別的實際資料筆數。
	- No：23 筆
	- Yes：25 筆
5. accuracy：準確率
6. macro avg：各項類別平均
7. weighted avg：各項類別依照樣本數加權計算

![](./images/問號.png)

## 作業：臺灣大專校院資料清理與前處理

本次作業使用政府資料開放平臺提供的「大專校院各校科系別概況」，透過 Pandas 與 Scikit-learn，練習資料清理、缺失值處理、類別編碼、特徵縮放，以及資料處理結果的解讀。

- 資料集：[大專校院各校科系別概況](https://data.gov.tw/dataset/9621)
- 資料來源：教育部統計處
- 檔案格式：CSV
- 本次作業不需要建立機器學習模型。

資料集包含哪些欄位？

| 欄位       | 類型 | 可以練習的內容          |
| -------- | -- | ---------------- |
| 學校名稱     | 類別 | EDA、資料篩選         |
| 科系名稱     | 類別 | EDA、資料篩選         |
| 縣市名稱     | 類別 | One-Hot Encoding |
| 體系別      | 類別 | One-Hot Encoding |
| 等級別      | 類別 | 類別編碼             |
| 學生數      | 數值 | 缺失值填補、特徵縮放       |
| 教師數      | 數值 | 缺失值填補、特徵縮放       |
| 上學年度畢業生數 | 數值 | 統計分析、Target 練習   |

以上欄位均列於官方資料集說明，實際缺失值情況需下載後檢查。

學生必須按照以下順序完成

```text
1. Pandas 讀取政府開放資料
        ↓
2. 探索資料 (EDA)
   (head / info / describe / shape)
        ↓
3. 資料清理
   (缺失值 / 重複值 / 資料型態)
        ↓
4. 選擇 Features / Target
        ↓
5. Train / Test Split
   (80% / 20%)
        ↓
6. SimpleImputer
   (中位數 / 眾數)
        ↓
7. One-Hot Encoding
   (縣市名稱 / 體系別)
        ↓
8. StandardScaler
   (學生數 / 教師數)
        ↓
9. MinMaxScaler
   (比較不同縮放方式)
        ↓
10. 視覺化與結果解讀
```


### 任務 1. 讀取資料

請匯入 `pandas`、`numpy`，使用 `pd.read_csv()` 讀取 `college.csv`，將資料存入 `df`。顯示前 5 筆資料與資料總筆數。

In [ ]:
# 請在這裡撰寫程式碼

### 任務 2. 探索資料 (EDA)

使用 `head()`、`shape`、`info()`、`describe()`、`columns.tolist()` 查看資料。請顯示所有欄位名稱，確認後續要用的欄位是否存在。

In [ ]:
# 請在這裡撰寫程式碼

### 任務 3. 檢查缺失值

請使用 `isna().sum()` 統計每個欄位的缺失筆數，再計算缺失比例 (%)，整理成 DataFrame，依缺失筆數由大到小排序並顯示前 10 個欄位。

In [ ]:
# 請在這裡撰寫程式碼

### 任務 4. 檢查重複與資料型態

使用 `duplicated().sum()` 檢查完全重複的資料筆數；查看數值欄位是否真的為數值型態。若有文字數字，使用 `pd.to_numeric(errors="coerce")` 轉換。顯示清理前後的資料型態。

In [ ]:
# 請在這裡撰寫程式碼

### 任務 5. 清理資料

建立 `df_clean = df.copy()`。處理完全重複資料，檢查學生數、教師數是否出現負數或其他不合理值，將確認無效的值轉為缺失值。顯示清理前後筆數及缺失值數量。請勿任意刪除有意義的 0。

In [ ]:
# 請在這裡撰寫程式碼

### 任務 6. 選擇 Features 與 Target

假設未來想研究「上學年度畢業生數」：請將可於預測時取得的學生數、教師數、縣市名稱、體系別選為 `X`，將上學年度畢業生數選為 `y`。不要將 Target 放入 X。先刪除 Target 缺失的列，顯示 X、y 的 shape 與前 5 筆。

In [ ]:
# 請在這裡撰寫程式碼


### 任務 7. 切分 Train / Test

使用 `train_test_split()`，設定 `test_size=0.2`、`random_state=42`，切出 `X_train`、`X_test`、`y_train`、`y_test`。顯示各自筆數，確認 X 與 y 對應。請先保留 `X_train_before` 與 `X_test_before` 複本。

In [ ]:
# 請在這裡撰寫程式碼


### 任務 8. 數值欄位補值

使用 `SimpleImputer(strategy="median")`，針對 `學生數`、`教師數` 的缺失值補值。訓練集使用 `fit_transform()`，測試集使用 `transform()`。顯示各欄中位數、補值前後缺失值數量，以及至少 3 筆補值前後比較 (若原始資料沒有缺失值，請如實說明)。

In [ ]:
# 請在這裡撰寫程式碼

### 任務 10. One-Hot Encoding

使用 `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`，將 `縣市名稱`、`體系別` 轉成 0/1 欄位。顯示編碼前 5 筆、編碼後 5 筆、編碼後欄位名稱與欄位數量。請將結果合併回 X_train 與 X_test，兩者欄位必須一致。

In [ ]:
# 請在這裡撰寫程式碼

### 任務 11. StandardScaler

使用 `StandardScaler()` 對學生數、教師數進行標準化。只在訓練集 `fit`。建立獨立的標準化結果，不要覆蓋尚未縮放的數值基準。顯示標準化前後各欄的平均數與標準差。

In [ ]:
# 請在這裡撰寫程式碼

### 任務 12. MinMaxScaler

從與上一題**相同的未縮放資料**出發，使用 `MinMaxScaler()` 對學生數、教師數進行 0～1 縮放。顯示縮放前後的最小值與最大值，並觀察測試集是否可能超出 0～1。

In [ ]:
# 請在這裡撰寫程式碼

### 任務 13. 視覺化與比較

使用 Matplotlib 繪製一張圖，比較學生數原始分布與 StandardScaler 處理後的分布 (可用直方圖)。標示圖表標題與座標軸名稱。再以 DataFrame 顯示原始值、標準化值、MinMax 值前 5 筆。

In [ ]:
# 請在這裡撰寫程式碼

<p style="text-align: center;">Copyright © 2026 Wei-Cheng Chen</p>